# HW1

In [1]:
import torch
import numpy as np
import pandas as pd

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False

if not torch.cuda.is_available():
    raise RuntimeError(":(")

device = torch.device("cuda:0")

MEMORY_FRACTION = 0.25
torch.cuda.set_per_process_memory_fraction(MEMORY_FRACTION, 0)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

PyTorch: 2.11.0+cu128
GPU: Tesla T4
CUDA: 12.8


## Модель


In [2]:
from collections import OrderedDict

import torch
from torch import nn


class myCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            OrderedDict(
                [
                    ("conv1", nn.Conv2d(3, 32, 7, stride=2, padding=3, bias=False)),
                    ("relu1", nn.ReLU(inplace=True)),
                    ("pool", nn.MaxPool2d(3, stride=2, padding=1)),
                    ("conv2", nn.Conv2d(32, 64, 5, padding=2, bias=False)),
                    ("relu2", nn.ReLU(inplace=True)),
                    ("conv3", nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False)),
                    ("relu3", nn.ReLU(inplace=True)),
                    ("conv4", nn.Conv2d(128, 256, 1, padding=0, bias=False)),
                    ("relu4", nn.ReLU(inplace=True)),
                    ("conv5", nn.Conv2d(256, 256, 3, stride=2, padding=1, bias=False)),
                    ("relu5", nn.ReLU(inplace=True)),
                    ("conv6", nn.Conv2d(256, 512, 1, padding=0, bias=False)),
                    ("relu6", nn.ReLU(inplace=True)),
                ]
            )
        )
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            OrderedDict(
                [
                    ("linear1", nn.Linear(512, 256)),
                    ("relu", nn.ReLU(inplace=True)),
                    ("linear2", nn.Linear(256, 100)),
                ]
            )
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


In [3]:
model = myCNN().to(device=device, dtype=torch.float32).eval()
print(model)
print("Параметров:", sum(p.numel() for p in model.parameters()))


myCNN(
  (features): Sequential(
    (conv1): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (relu1): ReLU(inplace=True)
    (pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (conv2): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), bias=False)
    (relu2): ReLU(inplace=True)
    (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (relu3): ReLU(inplace=True)
    (conv4): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (relu4): ReLU(inplace=True)
    (conv5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (relu5): ReLU(inplace=True)
    (conv6): Conv2d(256, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (relu6): ReLU(inplace=True)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=1)
  (classifier): Sequential(
    (linear1): Linear(in_features=512, out_features=256, bias=True)
    (relu): ReLU(inpla

## Проверка размеров

In [4]:
B, S = 2, 128
x = torch.randn(B, 3, S, S, device=device, dtype=torch.float32)
with torch.inference_mode():
    z = x
    print("input:  ", tuple(z.shape))
    for name, layer in model.features.named_children():
        z = layer(z)
        print(f"{name:8s}", tuple(z.shape))
    z = model.avgpool(z)
    print("avgpool:", tuple(z.shape))
    z = torch.flatten(z, 1)
    for name, layer in model.classifier.named_children():
        z = layer(z)
        print(f"{name:8s}", tuple(z.shape))
    y = model(x)
assert y.shape == (B, 100)
print("Итог:", tuple(y.shape), y.dtype, y.device)
del x, y, z


input:   (2, 3, 128, 128)
conv1    (2, 32, 64, 64)
relu1    (2, 32, 64, 64)
pool     (2, 32, 32, 32)
conv2    (2, 64, 32, 32)
relu2    (2, 64, 32, 32)
conv3    (2, 128, 16, 16)
relu3    (2, 128, 16, 16)
conv4    (2, 256, 16, 16)
relu4    (2, 256, 16, 16)
conv5    (2, 256, 8, 8)
relu5    (2, 256, 8, 8)
conv6    (2, 512, 8, 8)
relu6    (2, 512, 8, 8)
avgpool: (2, 512, 1, 1)
linear1  (2, 256)
relu     (2, 256)
linear2  (2, 100)
Итог: (2, 100) torch.float32 cuda:0


## Измерение прямого прохода

In [5]:
import gc
import random
import time

rng = random.Random(42)
base_sizes = [32, 64, 128, 224, 256, 384, 512]
base_batches = [1, 2, 4, 8, 16, 32, 64, 128, 256]
sizes = sorted(base_sizes + rng.sample([s for s in range(32, 513, 16) if s not in base_sizes], 4))
batches = sorted(base_batches + rng.sample([b for b in range(1, 257) if b not in base_batches], 3))

rows = []
for size in sizes:
    for batch in batches:
        row = dict(image_size=size, batch=batch, status="ok", latency_s=np.nan,
                   memory_bytes=np.nan, energy_j=np.nan,
                   is_validation=size not in base_sizes or batch not in base_batches)
        x = y = None
        try:
            gc.collect()
            torch.cuda.empty_cache()
            x = torch.randn(batch, 3, size, size, device=device, dtype=torch.float32)
            with torch.inference_mode():
                for _ in range(5):
                    y = model(x)
                    del y
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()
                y = model(x)
                torch.cuda.synchronize()
                row["memory_bytes"] = torch.cuda.max_memory_allocated()
                del y
                times = []
                for _ in range(20):
                    torch.cuda.synchronize()
                    start = time.perf_counter()
                    y = model(x)
                    torch.cuda.synchronize()
                    times.append(time.perf_counter() - start)
                    del y
                row["latency_s"] = float(np.median(times))
        except torch.cuda.OutOfMemoryError:
            row.update(status="OOM", latency_s=np.nan, memory_bytes=np.nan)
        finally:
            x = y = None
            gc.collect()
            torch.cuda.empty_cache()
        rows.append(row)
        print(f"S={size:3d}, B={batch:3d}: {row['status']}")

measurements = pd.DataFrame(rows)
display(measurements)
measurements.to_csv("measurements.csv", index=False)


S= 32, B=  1: ok
S= 32, B=  2: ok
S= 32, B=  4: ok
S= 32, B=  8: ok
S= 32, B= 16: ok
S= 32, B= 32: ok
S= 32, B= 64: ok
S= 32, B= 65: ok
S= 32, B= 70: ok
S= 32, B= 78: ok
S= 32, B=128: ok
S= 32, B=256: ok
S= 48, B=  1: ok
S= 48, B=  2: ok
S= 48, B=  4: ok
S= 48, B=  8: ok
S= 48, B= 16: ok
S= 48, B= 32: ok
S= 48, B= 64: ok
S= 48, B= 65: ok
S= 48, B= 70: ok
S= 48, B= 78: ok
S= 48, B=128: ok
S= 48, B=256: ok
S= 64, B=  1: ok
S= 64, B=  2: ok
S= 64, B=  4: ok
S= 64, B=  8: ok
S= 64, B= 16: ok
S= 64, B= 32: ok
S= 64, B= 64: ok
S= 64, B= 65: ok
S= 64, B= 70: ok
S= 64, B= 78: ok
S= 64, B=128: ok
S= 64, B=256: ok
S=112, B=  1: ok
S=112, B=  2: ok
S=112, B=  4: ok
S=112, B=  8: ok
S=112, B= 16: ok
S=112, B= 32: ok
S=112, B= 64: ok
S=112, B= 65: ok
S=112, B= 70: ok
S=112, B= 78: ok
S=112, B=128: ok
S=112, B=256: ok
S=128, B=  1: ok
S=128, B=  2: ok
S=128, B=  4: ok
S=128, B=  8: ok
S=128, B= 16: ok
S=128, B= 32: ok
S=128, B= 64: ok
S=128, B= 65: ok
S=128, B= 70: ok
S=128, B= 78: ok
S=128, B=128: 

,image_size,batch,status,latency_s,memory_bytes,energy_j,is_validation
0,32,1,ok,0.000683,1.876378e+07,NaN,False
1,32,2,ok,0.000742,1.887437e+07,NaN,False
2,32,4,ok,0.000748,1.909555e+07,NaN,False
3,32,8,ok,0.000774,1.953792e+07,NaN,False
4,32,16,ok,0.000978,1.484390e+07,NaN,False
...,...,...,...,...,...,...,...
127,512,65,ok,0.088498,1.798406e+09,NaN,True
128,512,70,ok,0.150049,1.844543e+09,NaN,True
129,512,78,ok,0.162979,1.920041e+09,NaN,True
130,512,128,ok,0.258928,3.092349e+09,NaN,False


## Энергия

In [6]:
%pip -q install nvidia-ml-py


In [7]:
import pynvml as nv

nv.nvmlInit()
handle = nv.nvmlDeviceGetHandleByIndex(0)
for index, row in measurements[measurements.status == "ok"].iterrows():
    x = y = None
    try:
        x = torch.randn(int(row.batch), 3, int(row.image_size), int(row.image_size),
                        device=device, dtype=torch.float32)
        with torch.inference_mode():
            start = time.perf_counter()
            while time.perf_counter() - start < 1:
                y = model(x)
                torch.cuda.synchronize()
                del y
            samples = [(time.perf_counter(), nv.nvmlDeviceGetPowerUsage(handle) / 1000)]
            count = 0
            while time.perf_counter() - samples[0][0] < 3:
                y = model(x)
                torch.cuda.synchronize()
                del y
                count += 1
                if time.perf_counter() - samples[-1][0] >= 0.05:
                    samples.append((time.perf_counter(), nv.nvmlDeviceGetPowerUsage(handle) / 1000))
            samples.append((time.perf_counter(), nv.nvmlDeviceGetPowerUsage(handle) / 1000))
            t, power = np.array(samples).T
            joules = np.sum(np.diff(t) * (power[:-1] + power[1:]) / 2) / count
            measurements.loc[index, "energy_j"] = joules
            print(f"S={int(row.image_size)}, B={int(row.batch)}: {joules:.6f} J")
    except (torch.cuda.OutOfMemoryError, nv.NVMLError) as error:
        print("Энергия не измерена:", error)
    finally:
        x = y = None
        gc.collect()
        torch.cuda.empty_cache()
    measurements.to_csv("measurements.csv", index=False)
nv.nvmlShutdown()


S=32, B=1: 0.028594 J
S=32, B=2: 0.032538 J
S=32, B=4: 0.030045 J
S=32, B=8: 0.032204 J
S=32, B=16: 0.035978 J
S=32, B=32: 0.045647 J
S=32, B=64: 0.062475 J
S=32, B=65: 0.063179 J
S=32, B=70: 0.108329 J
S=32, B=78: 0.106028 J
S=32, B=128: 0.166826 J
S=32, B=256: 0.303865 J
S=48, B=1: 0.028242 J
S=48, B=2: 0.032120 J
S=48, B=4: 0.034055 J
S=48, B=8: 0.039060 J
S=48, B=16: 0.041197 J
S=48, B=32: 0.053402 J
S=48, B=64: 0.089771 J
S=48, B=65: 0.079058 J
S=48, B=70: 0.133040 J
S=48, B=78: 0.147979 J
S=48, B=128: 0.225888 J
S=48, B=256: 0.422570 J
S=64, B=1: 0.035947 J
S=64, B=2: 0.037760 J
S=64, B=4: 0.038886 J
S=64, B=8: 0.043004 J
S=64, B=16: 0.041875 J
S=64, B=32: 0.071801 J
S=64, B=64: 0.108085 J
S=64, B=65: 0.115570 J
S=64, B=70: 0.207916 J
S=64, B=78: 0.218538 J
S=64, B=128: 0.357594 J
S=64, B=256: 0.688977 J
S=112, B=1: 0.044478 J
S=112, B=2: 0.041360 J
S=112, B=4: 0.041122 J
S=112, B=8: 0.066838 J
S=112, B=16: 0.097535 J
S=112, B=32: 0.181194 J
S=112, B=64: 0.344998 J
S=112, B=65: 0

## Формулы

In [8]:
import numpy as np


def flops(image_size, batch):
    s, b = np.asarray(image_size, float), np.asarray(batch, float)
    return b * (17714 * s**2 + 313700)


def memory(image_size, batch):
    s, b = np.asarray(image_size, float), np.asarray(batch, float)
    return 4161296 + 68 * b * s**2


def latency(image_size, batch, theta):
    s, b = np.asarray(image_size, float), np.asarray(batch, float)
    moved = 4 * (95 * b * s**2 + 2148 * b + 1040324)
    return theta['overhead_s'] + np.maximum(
        flops(image_size, batch) / theta['flops_per_s'],
        moved / theta['bytes_per_s'])


def energy(image_size, batch, theta_energy):
    return theta_energy['power_w'] * latency(
        image_size, batch, theta_energy['latency_theta'])


## Калибровка и графики

In [9]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

measurements = pd.read_csv('measurements.csv')
measurements['is_validation'] = (
    ~measurements.image_size.isin([32, 64, 128, 224, 256, 384, 512])
    | ~measurements.batch.isin([1, 2, 4, 8, 16, 32, 64, 128, 256]))
if 'energy_j' not in measurements:
    measurements['energy_j'] = np.nan
train = measurements[(measurements.status == 'ok') & ~measurements.is_validation]
s, b = train.image_size.to_numpy(), train.batch.to_numpy()
names = ['overhead_s', 'flops_per_s', 'bytes_per_s']
fit = least_squares(
    lambda x: np.log(latency(s, b, dict(zip(names, 10**x))) / train.latency_s),
    [-3, 12, 10], bounds=([-9, 7, 5], [0, 16, 15]))
theta = dict(zip(names, 10**fit.x))
theta_energy = None
measured_energy = train[train.energy_j > 0]
if len(measured_energy):
    t = latency(measured_energy.image_size, measured_energy.batch, theta)
    power = np.dot(t, measured_energy.energy_j) / np.dot(t, t)
    theta_energy = {'power_w': float(power), 'latency_theta': theta}

Path('results/figures').mkdir(parents=True, exist_ok=True)
measurements.to_csv('results/measurements.csv', index=False)
Path('results/theta.json').write_text(json.dumps({'latency': theta, 'energy': theta_energy}, indent=2))
metrics = [('latency_s', lambda s, b: latency(s, b, theta), 'Time, s'),
           ('memory_bytes', memory, 'Memory, bytes')]
if theta_energy:
    metrics.append(('energy_j', lambda s, b: energy(s, b, theta_energy), 'Energy, J'))
for column, predict, unit in metrics:
    fig, ax = plt.subplots(figsize=(5, 5))
    for validation, label in [(False, 'Calibration'), (True, 'Validation')]:
        part = measurements[(measurements.status == 'ok') & (measurements.is_validation == validation)
                            & (measurements[column] > 0)]
        ax.scatter(predict(part.image_size, part.batch), part[column], s=18, label=label)
    values = measurements.loc[(measurements.status == 'ok') & (measurements[column] > 0), column]
    ax.plot([values.min(), values.max()], [values.min(), values.max()], 'k-', label='Exact match')
    ax.set(xlabel=f'Predicted {unit}', ylabel=f'Measured {unit}', xscale='log', yscale='log')
    ax.legend()
    fig.tight_layout()
    fig.savefig(f'results/figures/{column}.png', dpi=150)
    plt.close(fig)
    for validation in [False, True]:
        part = measurements[(measurements.status == 'ok') & (measurements.is_validation == validation)
                            & (measurements[column] > 0)]
        if len(part):
            error = np.mean(np.abs(predict(part.image_size, part.batch)/part[column] - 1))*100
            print(column, 'validation' if validation else 'train', f'MAPE={error:.2f}%')
print('OOM:', measurements.loc[measurements.status == 'OOM', ['image_size', 'batch']].values.tolist())


latency_s train MAPE=30.60%
latency_s validation MAPE=24.50%
memory_bytes train MAPE=41.88%
memory_bytes validation MAPE=38.08%
energy_j train MAPE=50.11%
energy_j validation MAPE=33.38%
OOM: [[496, 256], [512, 256]]


## Сохранение результатов

In [10]:
limit = torch.cuda.get_device_properties(0).total_memory * MEMORY_FRACTION
print(f"GPU: {torch.cuda.get_device_name(0)}, лимит: {limit / 2**30:.3f} GiB")
for row in measurements[measurements.status == "OOM"].itertuples():
    predicted = memory(row.image_size, row.batch)
    print(f"S={row.image_size}, B={row.batch}: {predicted / 2**30:.3f} GiB, "
          f"оценка выше лимита: {predicted > limit}")

import shutil
from google.colab import files
files.download(shutil.make_archive("hw1_results", "zip", "results"))


GPU: Tesla T4, лимит: 3.641 GiB
S=496, B=256: 3.992 GiB, оценка выше лимита: True
S=512, B=256: 4.254 GiB, оценка выше лимита: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>